In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential, Model
from keras.layers import LSTM, Dense, Dropout, Input, RepeatVector
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix, classification_report, matthews_corrcoef
import matplotlib.pyplot as plt
import scikitplot as skplt
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import LSTM, GRU
from keras.layers import Dense , Dropout
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc
from sklearn.multiclass import OneVsRestClassifier
from itertools import cycle
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
import scikitplot as skplt
from sklearn.preprocessing import label_binarize
from sklearn.datasets import load_digits
#from yellowbrick.features import ParallelCoordinates
from keras.layers import Bidirectional, LSTM
from keras.layers import Bidirectional, GRU
from sklearn.metrics import matthews_corrcoef

In [ ]:
filepath= "C:/Users/Theda/Desktop/6G Driven SHS/Edge-iiotset.csv"

In [ ]:
mydata= pd.read_csv(filepath)

In [ ]:
mydata.info()

In [ ]:
drop_columns = ["frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4","arp.dst.proto_ipv4", 

         "http.file_data","http.request.full_uri","icmp.transmit_timestamp",

         "http.request.uri.query", "tcp.options","tcp.payload","tcp.srcport",

         "tcp.dstport", "udp.port", "mqtt.msg"]

mydata.drop(drop_columns, axis=1, inplace=True)


In [ ]:
print(mydata['Attack_type'].value_counts())

In [ ]:
mydata.drop('Attack_label', axis='columns', inplace=True)


In [ ]:
mydata.replace([np.inf, -np.inf], np.nan, inplace=True)
mydata.dropna(inplace=True)
mydata.drop_duplicates(inplace=True)
mydata.reset_index(inplace=True, drop=True)
mydata = mydata.sample(frac=1).reset_index(drop=True)

In [ ]:
mydata.info()

In [ ]:
attacks = {'Normal': 0 ,'DDoS_UDP' :1, 'DDoS_ICMP':2, 'SQL_injection':3, 'Password':4,
       'Vulnerability_scanner':5 , 'DDoS_TCP':6, 'DDoS_HTTP':7, 'Uploading':8, 'Backdoor':9, 
       'Port_Scanning':10, 'XSS':11, 'Ransomware':12, 'Fingerprinting':13, 'MITM':14}
mydata['label']=mydata['Attack_type'].map(attacks)

In [ ]:
print(mydata['label'].value_counts())

In [ ]:
mydata.drop('Attack_type', axis='columns', inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()

In [ ]:
mydata['mqtt.protoname'] = mydata['mqtt.protoname'].astype(str)
mydata['mqtt.topic'] = mydata['mqtt.topic'].astype(str)
mydata['mqtt.conack.flags'] = mydata['mqtt.conack.flags'].astype(str)
mydata['dns.qry.name.len'] = mydata['dns.qry.name.len'].astype(str)
mydata['http.request.method'] = mydata['http.request.method'].astype(str)
mydata['http.referer'] = mydata['http.referer'].astype(str)
mydata['http.request.version'] = mydata['http.request.version'].astype(str)

In [ ]:
from sklearn.preprocessing import LabelEncoder

labelencoder = LabelEncoder()

mydata['mqtt.protoname'] = labelencoder.fit_transform(mydata['mqtt.protoname'])
mydata['mqtt.topic'] = labelencoder.fit_transform(mydata['mqtt.topic'])
mydata['mqtt.conack.flags'] = labelencoder.fit_transform(mydata['mqtt.conack.flags'])
mydata['dns.qry.name.len'] = labelencoder.fit_transform(mydata['dns.qry.name.len'])
mydata['http.request.method'] = labelencoder.fit_transform(mydata['http.request.method'])
mydata['http.referer'] = labelencoder.fit_transform(mydata['http.referer'])
mydata['http.request.version'] = labelencoder.fit_transform(mydata['http.request.version'])


In [ ]:
mydata.info()

In [ ]:
X = mydata.iloc[:, 0:46]
Y = mydata.iloc[:, 46:47]

In [ ]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X = sc.fit_transform(X)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.30, random_state=42)
x_train = np.array(x_train)
x_train = np.reshape (x_train,(x_train.shape[0], 1, x_train.shape[1]))
#Reshape and normalize test data
x_test = np.array(x_test)
x_test = np.reshape (x_test,(x_test.shape[0], 1, x_test.shape[1]))

In [ ]:
Num_classes = len(np.unique(Y))
y_train_ohe=to_categorical(y_train,Num_classes)
y_train_ohe = pd.DataFrame(y_train_ohe)

In [ ]:
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
import numpy as np
import tensorflow as tf

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Attention, Input, Dense, LSTM, Bidirectional, Conv1D, LayerNormalization, Add, Dropout, Concatenate, Flatten
from tensorflow.keras.models import Model

# Enhanced Multi-Scale Multi-Head Attention Layer
class MultiScaleMultiHeadAttention(Layer):
    def __init__(self, num_heads=4, scales=[1, 2, 4], feature_dim=None, **kwargs):
        super(MultiScaleMultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.scales = scales
        self.conv_layers = [
            [Conv1D(filters=feature_dim, kernel_size=scale, padding='same', activation='relu')
             for _ in range(num_heads)] for scale in scales]
        self.attention_heads = [[Attention() for _ in range(num_heads)] for _ in self.scales]
        self.fusion_layer = Dense(feature_dim * len(scales) * num_heads, activation='sigmoid')

    def call(self, x):
        multi_scale_outputs = []
        for scale_idx, (scale_heads, scale_conv_layers) in enumerate(zip(self.attention_heads, self.conv_layers)):
            for head_idx, (attention, conv) in enumerate(zip(scale_heads, scale_conv_layers)):
                conv_x = conv(x)
                attention_output = attention([conv_x, conv_x])
                multi_scale_outputs.append(attention_output)
        
        concatenated = Concatenate(axis=-1)(multi_scale_outputs)
        fusion_weight = self.fusion_layer(concatenated)
        fused_output = concatenated * fusion_weight
        return fused_output
        
# Residual BiLSTM block with matching dimensions
def residual_bilstm_block(x, units):
    x_res = x  # Save residual connection
    x = Bidirectional(LSTM(units, activation='relu', return_sequences=True))(x)
    x = LayerNormalization()(x)
    
    # Match dimensions of residual connection using a Dense layer
    x_res = Dense(units * 2)(x_res)  # Since Bidirectional doubles the units
    x = Add()([x, x_res])  # Adding the residual connection
    return x

# Assuming x_train_encoded.shape[2] gives the number of features per timestep
num_features = x_train.shape[2]  # You should define x_train_encoded.shape[2] before this script

# Define the enhanced model with Multi-Scale Temporal Attention and Adaptive Feature Fusion
inputs = Input(shape=(x_train.shape[1], x_train.shape[2]))
x = Bidirectional(LSTM(80, activation='relu', return_sequences=True))(inputs)

# Adding residual BiLSTM blocks with dimension matching
x = residual_bilstm_block(x, 32)
x = residual_bilstm_block(x, 16)

x = Dropout(0.2)(x)
# Multi-Scale Multi-Head Attention
x = MultiScaleMultiHeadAttention(num_heads=4, scales=[1, 2, 4], feature_dim=num_features)(x)
x = Flatten()(x)  # Flatten to ensure the output dimension is correct
x = Dropout(rate=0.2)(x)
outputs = Dense(15, activation='softmax')(x)  # Update Num_classes to your actual number of classes

model = Model(inputs=inputs, outputs=outputs)
# Compile with AdamW and custom learning rate schedule
model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.AdamW(learning_rate=2e-4, weight_decay=1e-4), metrics=['accuracy'])

# Custom learning rate schedule
lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

In [ ]:
history = model.fit(x_train, y_train_ohe, epochs=20, validation_split=0.5, batch_size=64, callbacks=[lr_schedule])

In [ ]:
epochs = len(history.history['accuracy'])
plt.plot(history.history['accuracy'], label='Training Accuracy', color="green", marker='*')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color="red", marker='*')
# plt.title('Model loss')  # Commented out as it seems you don't want the title
plt.ylabel('Model Accuracy', fontsize=16)
plt.xlabel('No. of Epochs', fontsize=16)
plt.rcParams['figure.dpi'] = 300
plt.legend()
plt.grid()  # Commented out as it seems you don't want the grid
# Set the ticks on the x-axis to 2, 4, and 6
plt.xticks(np.arange(0, epochs, step=2), fontsize=16)  # Adjust the step size as needed
plt.yticks(fontsize=16)
# plt.rcParams['figure.figsize'] = [6.0, 4.0]  # Commented out as it seems you have another setting for figure size
plt.savefig('edgeacc1.png', format='png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
epochs = len(history.history['loss'])
plt.plot(history.history['loss'], label='Training Loss', color="green", marker='*')
plt.plot(history.history['val_loss'], label='Validation Loss', color="red", marker='*')
# plt.title('Model loss')  # Commented out as it seems you don't want the title
plt.ylabel('Model Loss', fontsize=16)
plt.xlabel('No. of Epochs', fontsize=16)
plt.rcParams['figure.dpi'] = 300
plt.legend()
plt.grid()  # Commented out as it seems you don't want the grid
# Set the ticks on the x-axis to 2, 4, and 6
plt.xticks(np.arange(0, epochs, step=2), fontsize=16)  # Adjust the step size as needed
plt.yticks(fontsize=16)
# plt.rcParams['figure.figsize'] = [6.0, 4.0]  # Commented out as it seems you have another setting for figure size
plt.savefig('edgeloss1.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
predict_prob=model.predict([x_test])
predict_classes=np.argmax(predict_prob,axis=1)
prediction = model.predict(x_test)
prediction =prediction.argmax(1)

In [ ]:
from sklearn.preprocessing import label_binarize
# Assuming y_test and predict_prob are already defined
# Binarize the output
y_test_binarized = label_binarize(y_test, classes=np.arange(Num_classes))
# Class names (modify these names as per your actual class names)
class_names = ['Normal','DDoS_UDP', 'DDoS_ICMP', 'SQL_injection', 'Password',
       'Vulnerability_scanner', 'DDoS_TCP', 'DDoS_HTTP', 'Uploading', 
       'Backdoor','Port_Scanning', 'XSS', 'Ransomware', 
       'Fingerprinting', 'MITM']
# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(Num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], predict_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
# Compute micro-average ROC curve and ROC area
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_binarized.ravel(), predict_prob.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
# Compute macro-average ROC curve and ROC area
# First aggregate all false positive rates
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(Num_classes)]))
# Then interpolate all ROC curves at this points
mean_tpr = np.zeros_like(all_fpr)
for i in range(Num_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
# Finally average it and compute AUC
mean_tpr /= Num_classes
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])
# Plot ROC curves for each class, micro-average, and macro-average
plt.figure(figsize=(7, 5))  # Increase figure size for better visibility
# Plot micro-average ROC curve
plt.plot(fpr["micro"], tpr["micro"],
         label=f'micro-average ROC curve (area = {roc_auc["micro"]:.4f})',
         color='deeppink', linestyle=':', linewidth=4)
# Plot macro-average ROC curve
plt.plot(fpr["macro"], tpr["macro"],
         label=f'macro-average ROC curve (area = {roc_auc["macro"]:.4f})',
         color='navy', linestyle=':', linewidth=4)
# Plot ROC curve for each class
for i in range(Num_classes):
    plt.plot(fpr[i], tpr[i], label=f'ROC Curve of {class_names[i]} (area = {roc_auc[i]:.4f})')
# Plot the diagonal line
#plt.plot([0, 1], [0, 1], 'k--')
# Set the limits from 0.9 to 1.0
plt.xlim([0.0, 1.0])
plt.ylim([0.95, 1.001])
plt.xlabel('False Positive Rate', fontsize=16)
plt.ylabel('True Positive Rate', fontsize=16)
#plt.title('Receiver Operating Characteristic (Zoomed In)')
# Place the legend inside the plot area
plt.legend(loc='lower right', fontsize=10.5, title_fontsize='xx-large')
plt.grid()
plt.tight_layout()  # Adjust subplots to give space for the legend
plt.savefig('edgeroc.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("Accuracy:" + str(accuracy_score(y_test, prediction)* 100)) 
print(precision_score(y_test, prediction, average="macro"))  
print(recall_score(y_test, prediction, average="macro"))
print(f1_score(y_test, prediction, average="macro"))

In [ ]:
print(precision_score(y_test, prediction, average="weighted"))  
print(recall_score(y_test, prediction, average="weighted"))
print(f1_score(y_test, prediction, average="weighted"))

In [ ]:
print(classification_report(y_test, prediction))
cm = confusion_matrix(y_test, prediction)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Class names
class_names = ['Normal', 'DDoS_UDP', 'DDoS_ICMP', 'SQL_injection', 'Password',
               'Vulnerability_scanner', 'DDoS_TCP', 'DDoS_HTTP', 'Uploading',
               'Backdoor', 'Port_Scanning', 'XSS', 'Ransomware',
               'Fingerprinting', 'MITM']

# Assuming y_test and prediction are defined elsewhere
# Calculate the confusion matrix
cm = confusion_matrix(y_test, prediction)
# Create a custom colormap
cmap = sns.color_palette("YlOrBr", as_cmap=True)  # Use a yellow-orange-brown gradient
# Plot the confusion matrix using seaborn
plt.figure(figsize=(18, 14))
ax = sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=class_names, yticklabels=class_names, cbar=False)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.xlabel('Predicted labels', fontsize=16)
plt.ylabel('True labels', fontsize=16)
#plt.title('Confusion Matrix')
plt.tight_layout()  # Adjust layout to make room for rotated x labels
plt.savefig('edgecm.png', format='png', dpi=300, bbox_inches='tight')
plt.rcParams['figure.dpi'] = 300
plt.show()

# Set the resolution of the figure


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Class names
class_names = ['Normal', 'DDoS_UDP', 'DDoS_ICMP', 'SQL_injection', 'Password',
               'Vulnerability_scanner', 'DDoS_TCP', 'DDoS_HTTP', 'Uploading',
               'Backdoor', 'Port_Scanning', 'XSS', 'Ransomware',
               'Fingerprinting', 'MITM']

# Assuming y_test and prediction are defined elsewhere
# Calculate the confusion matrix
cm = confusion_matrix(y_test, prediction)

# Normalize the confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Create a custom colormap
cmap = sns.color_palette("YlOrBr", as_cmap=True)  # Use a yellow-orange-brown gradient

# Plot the normalized confusion matrix using seaborn
plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap=cmap, xticklabels=class_names, yticklabels=class_names, cbar=False)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.xlabel('Predicted labels', fontsize=16)
plt.ylabel('True labels', fontsize=16)
#plt.title('Normalized Confusion Matrix')
plt.tight_layout()  # Adjust layout to make room for rotated x labels

# Print the normalized confusion matrix for reference
print("Normalized Confusion Matrix:\n", cm_normalized)

# Set the resolution of the figure for better output quality
plt.rcParams['figure.dpi'] = 300
plt.savefig('edgecmnormalize.png', format='png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
FP = cm.sum(axis=0) - np.diag(cm)  
FN = cm.sum(axis=1) - np.diag(cm)
TP = np.diag(cm)
TN = cm.sum() - (FP + FN + TP)
FP = FP.astype(float)
FN = FN.astype(float)
TP = TP.astype(float)
TN = TN.astype(float)
TPR = TP/(TP+FN)
TNR = TN/(TN+FP) 
FPR = FP/(FP+TN)
FNR = FN/(TP+FN)
FDR = FP/(TP+FP)
FOR = FN/(FN+TN)
PPV = TP/(TP+FP)
NPV = TN/(TN+FN)
# Overall accuracy
ACC = (TP+TN)/(TP+FP+FN+TN)
Precision  = TP / (FP + TP)
Recall = TP / (FN + TP)
F1 = 2* Precision * Recall/ (Precision + Recall )
MCC=matthews_corrcoef(y_test, prediction)

In [ ]:
FDR = FP / (TP + FP + 1e-10)
PPV = TP / (TP + FP + 1e-10)
Precision = TP / (FP + TP + 1e-10)

In [ ]:
Precision

In [ ]:
Recall

In [ ]:
F1

In [ ]:
FPR

In [ ]:
TPR

In [ ]:
TNR

In [ ]:
FOR

In [ ]:
FDR

In [ ]:
FNR

In [ ]:
MCC

In [ ]:
import time
batch_size = 32
_ = model.predict(x_test, batch_size=batch_size)
num_runs = 5
total_time = 0
for _ in range(num_runs):
    start_time = time.time()
    _ = model.predict(x_test, batch_size=batch_size)
    total_time += time.time() - start_time
average_batch_prediction_time = total_time / num_runs
total_samples = x_test.shape[0]
average_inference_time_per_sample = average_batch_prediction_time / total_samples
print(f"Average Total Time for Batch Prediction (over {num_runs} runs): {average_batch_prediction_time:.4f} seconds")
print(f"Average Inference Time per Sample: {average_inference_time_per_sample:.6f} seconds")

In [ ]:
model.summary()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Layer, Attention, Input, Dense, LSTM, Bidirectional, Conv1D, LayerNormalization, Add, Dropout, Concatenate, Flatten
from tensorflow.keras.models import Model
import time

# Enhanced Multi-Scale Multi-Head Attention Layer
class MultiScaleMultiHeadAttention(Layer):
    def __init__(self, num_heads=4, scales=[1, 2, 4], feature_dim=None, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.scales = scales
        self.conv_layers = [
            [Conv1D(filters=feature_dim, kernel_size=scale, padding='same', activation='relu')
             for _ in range(num_heads)] for scale in scales]
        self.attention_heads = [[Attention() for _ in range(num_heads)] for _ in self.scales]
        self.fusion_layer = Dense(feature_dim * len(scales) * num_heads, activation='sigmoid')

    def call(self, inputs):
        multi_scale_outputs = []
        for scale_heads, scale_conv_layers in zip(self.attention_heads, self.conv_layers):
            for attention, conv in zip(scale_heads, scale_conv_layers):
                conv_x = conv(inputs)
                attention_output = attention([conv_x, conv_x], return_attention_scores=False)
                multi_scale_outputs.append(attention_output)
        concatenated = Concatenate(axis=-1)(multi_scale_outputs)
        fusion_weight = self.fusion_layer(concatenated)
        fused_output = concatenated * fusion_weight
        return fused_output

# Residual BiLSTM block with matching dimensions
def residual_bilstm_block(x, units):
    x_res = x  # Save residual connection
    x = Bidirectional(LSTM(units, return_sequences=True, activation='relu'))(x)
    x = LayerNormalization()(x)
    x_res = Dense(units * 2)(x_res)  # Since Bidirectional doubles the units
    x = Add()([x, x_res])  # Adding the residual connection
    return x

# Synthetic data
num_samples = 1000  # Number of samples for throughput testing
num_features = 128  # Number of features per sample
sequence_length = 50  # Length of each sequence

# Create synthetic data
x_train_encoded = np.random.randn(num_samples, sequence_length, num_features).astype(np.float32)

# Define the model
inputs = Input(shape=(sequence_length, num_features))
x = Bidirectional(LSTM(80, activation='relu', return_sequences=True))(inputs)
x = residual_bilstm_block(x, 32)
x = residual_bilstm_block(x, 16)
x = Dropout(0.2)(x)
x = MultiScaleMultiHeadAttention(num_heads=4, scales=[1, 2, 4], feature_dim=num_features)(x)
x = Flatten()(x)
x = Dropout(0.2)(x)
outputs = Dense(15, activation='softmax')(x)
model = Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.AdamW(learning_rate=2e-4, weight_decay=1e-4), metrics=['accuracy'])

# Measure throughput
start_time = time.time()
_ = model.predict(x_train_encoded, batch_size=10)
end_time = time.time()

# Calculate elapsed time and throughput
elapsed_time = end_time - start_time
throughput = num_samples / elapsed_time
print(f"Throughput: {throughput:.2f} samples/second")


In [ ]:
print(mydata['label'].value_counts())

In [ ]:
print(y_train['label'].value_counts())

In [ ]:
print(y_test['label'].value_counts())